# Q3-01：Q2 零点预测 + 06/12 概率场景 6 小时滚动更新

本 notebook 按以下口径回测：

1. 每日 00:00 使用 Q2 冻结预测模型生成未来 24 小时负荷、光伏及净负荷场景，并制定全天初始购电计划。
2. 时间推进到 06:00 与 12:00 时，只允许修改接下来 6 小时的购电计划。
3. 更新时只能使用当时已经发布的预测和已经观测到的时段；未来 6 小时的真实负荷、真实光伏只在事后执行/结算环节使用，绝不进入规划场景。
4. 费用、调整费、紧急购电费、储能约束和 baseline notebook 保持同一口径。

最后一个 cell 输出评价期总费用，方便和 baseline 的 `summary.csv` 对比。


**费用口径修正**：零点目标为“普通计划购电费 + 期望紧急购电费”；日内原计划费为常数，只优化调整费、紧急购电费及该方案启用的风险项。评价期统一为 2025-02-01 至 2025-12-31。使用 Q2 预测的场景调度不等同于 Q2 的 0.85 分位数固定调度策略。


In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix, csr_matrix, hstack

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

ROOT = Path.cwd()
while not (ROOT / 'C题.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

ACTUAL_PATH = ROOT / 'q2' / 'input' / 'q2_actual_load_pv.csv'
PRICE_PATH = ROOT / 'q2' / 'input' / 'q2_fixed_prices.csv'
OUTPUT_DIR = ROOT / '.tmp' / 'q3_01_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'interval_minutes': 10,
    'interval_hours': 1 / 6,
    'charge_efficiency': 0.9,
    'discharge_efficiency': 0.9,
    'storage_min_kwh': 1200.0,
    'storage_max_kwh': 10800.0,
    'initial_storage_kwh': 6000.0,
    'planned_terminal_kwh': 6000.0,
    'max_charge_power_kw': 5000.0,
    'max_discharge_power_kw': 5000.0,
    'emergency_multiplier': 5.0,
    'adjustment_up_multiplier': 1.5,
    'adjustment_down_multiplier': 0.5,
    'history_days': 28,
    'tolerance': 1e-6,
}

UPDATE_STRATEGIES = {
    '对照：00时Q2计划不更新': (),
    '方案：06时+12时历史误差场景更新未来6h': (6, 12),
}

print('输入文件：', ACTUAL_PATH, PRICE_PATH)
print('输出目录：', OUTPUT_DIR)


输入文件： /Users/jinyu/workspace/MCM/CUMCM_2026/q2/input/q2_actual_load_pv.csv /Users/jinyu/workspace/MCM/CUMCM_2026/q2/input/q2_fixed_prices.csv
输出目录： /Users/jinyu/workspace/MCM/CUMCM_2026/.tmp/q3_01_outputs


## 1. 读取实际数据并调用 Q2 冻结预测模型

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


actual = pd.read_csv(ACTUAL_PATH, parse_dates=['date']).sort_values(['date', 'slot'])
prices_df = pd.read_csv(PRICE_PATH).sort_values('slot')

require(len(actual) == 365 * 144, '实际数据应为 365 天 x 144 个 10 分钟区间')
require(actual.groupby('date').size().eq(144).all(), '实际数据存在日期缺行')
require(not actual.duplicated(['date', 'slot']).any(), '实际数据存在重复日期-时段')
require(len(prices_df) == 144, '固定电价应为 144 个时段')

dates = pd.DatetimeIndex(sorted(actual['date'].unique()))
load_kw = actual['load_kw'].to_numpy(float).reshape(len(dates), 144)
pv_kw = actual['pv_kw'].to_numpy(float).reshape(len(dates), 144)
prices = prices_df['price_yuan_per_kwh'].to_numpy(float)

require(np.isfinite(load_kw).all() and np.isfinite(pv_kw).all(), '实际功率含非有限值')
require(np.isfinite(prices).all() and (prices > 0).all(), '电价必须为正且有限')

import sys
sys.path.insert(0, str(ROOT / 'q2'))
from q2_final import FROZEN_FORECAST, forecast_scenarios

q2_pred_load_kw, q2_pred_pv_kw, q2_scenario_net_kw = forecast_scenarios(dates, load_kw, pv_kw)
q2_first_valid_day = 14
q2_first_scenario_day = 15

require(np.isfinite(q2_pred_load_kw[q2_first_valid_day:]).all(), 'Q2 负荷预测在有效期内含非有限值')
require(np.isfinite(q2_pred_pv_kw[q2_first_valid_day:]).all(), 'Q2 光伏预测在有效期内含非有限值')
require(all(k in q2_scenario_net_kw for k in range(q2_first_scenario_day, len(dates))), 'Q2 净负荷场景不完整')

print('Q2 冻结参数：', FROZEN_FORECAST)
print(f'读取 {len(dates)} 天，实际数据形状：{load_kw.shape}')


Q2 冻结参数： {'window_days': 60, 'k_load': 3, 'k_pv': 2, 'alpha_load': 0.3, 'alpha_pv': 10.0, 'scenario_days': 28}
读取 365 天，实际数据形状：(365, 144)


## 2. 预测场景、规划、调整与实际执行模型

In [3]:
def scenario_net_load(day_index, start_slot):
    '''返回从 start_slot 到 24:00 的可用净负荷场景，单位 kWh。

    场景由 Q2 预测和截至前一日形成的历史误差轨迹构成。若当天已经
    观测到若干时段，则只用这些已观测时段估计在线偏差；绝不读取
    当天 start_slot 之后的真实数据。
    '''
    if day_index in q2_scenario_net_kw:
        scenarios = q2_scenario_net_kw[day_index][:, start_slot:].copy() * CFG['interval_hours']
    else:
        # 预热期也不读取当天未来值：用上一日已知净负荷作持久性预测。
        if day_index > 0:
            prior_net = (load_kw[day_index - 1, start_slot:] - pv_kw[day_index - 1, start_slot:]) * CFG['interval_hours']
        else:
            prior_net = np.zeros(144 - start_slot)
        scenarios = prior_net[None, :]

    # 当前日已发生的误差可用于校正未来，但不能使用未来时段的真实值。
    if start_slot > 0 and day_index >= q2_first_valid_day:
        obs_l = load_kw[day_index, :start_slot] - q2_pred_load_kw[day_index, :start_slot]
        obs_p = pv_kw[day_index, :start_slot] - q2_pred_pv_kw[day_index, :start_slot]
        w = np.exp(np.linspace(-2.0, 0.0, start_slot))
        bias_l = np.average(obs_l, axis=0, weights=w)
        bias_p = np.average(obs_p, axis=0, weights=w)
        scenarios = scenarios + (bias_l - bias_p) * CFG['interval_hours']
        scenarios = np.maximum(scenarios, 0.0)
    return scenarios


def _storage_problem_parts(net_scenarios, p, initial_kwh, fixed_grid=None, lock_from=None):
    a = np.asarray(net_scenarios, float)
    p = np.asarray(p, float)
    s, n = a.shape
    ec, ed = CFG['charge_efficiency'], CFG['discharge_efficiency']
    mc = CFG['max_charge_power_kw'] * CFG['interval_hours']
    md = CFG['max_discharge_power_kw'] * CFG['interval_hours']
    lo, hi = CFG['storage_min_kwh'], CFG['storage_max_kwh']

    # x = [g(n), c(n), d(n), E(n+1), h(s*n), u(n), v(n)]
    h0 = 4 * n + 1
    u0 = h0 + s * n
    v0 = u0 + n
    m = v0 + n
    obj = np.zeros(m)
    # 零点计划须计入普通购电费；日内原计划费已固定，只优化调整费与紧急费。
    if fixed_grid is None:
        obj[:n] = p
    obj[h0:u0] = np.tile(CFG['emergency_multiplier'] * p / s, s)

    lower = np.zeros(m)
    upper = np.full(m, np.inf)
    upper[n:2*n] = mc
    upper[2*n:3*n] = md
    lower[3*n:4*n+1] = lo
    upper[3*n:4*n+1] = hi
    lower[3*n] = upper[3*n] = initial_kwh
    lower[4*n] = upper[4*n] = CFG['planned_terminal_kwh']

    t = np.arange(n)
    eq_rows = np.r_[t, t, t, t]
    eq_cols = np.r_[n+t, 2*n+t, 3*n+t, 3*n+t+1]
    eq_data = np.r_[-ec*np.ones(n), np.ones(n)/ed, -np.ones(n), np.ones(n)]
    storage_eq = coo_matrix((eq_data, (eq_rows, eq_cols)), shape=(n, m)).tocsr()

    scenario_rows = np.arange(s*n)
    scenario_slots = np.tile(t, s)
    scenario_ub = coo_matrix(
        (
            np.r_[-np.ones(s*n), np.ones(s*n), -np.ones(s*n), -np.ones(s*n)],
            (
                np.tile(scenario_rows, 4),
                np.r_[scenario_slots, n+scenario_slots, 2*n+scenario_slots, h0+scenario_rows],
            ),
        ),
        shape=(s*n, m),
    ).tocsr()
    scenario_rhs = -a.ravel()

    if fixed_grid is not None:
        base = np.asarray(fixed_grid, float)
        require(len(base) == n, '调整区间计划长度不一致')
        from scipy.sparse import vstack
        grid_eq = coo_matrix(
            (
                np.r_[np.ones(n), -np.ones(n), np.ones(n)],
                (np.r_[t, t, t], np.r_[t, u0+t, v0+t]),
            ),
            shape=(n, m),
        ).tocsr()
        all_eq = vstack([storage_eq, grid_eq]).tocsr()
        all_rhs = np.r_[np.zeros(n), base]
        obj[u0:v0] = CFG['adjustment_up_multiplier'] * p
        obj[v0:] = CFG['adjustment_down_multiplier'] * p
        if lock_from is not None:
            lower[lock_from:n] = base[lock_from:]
            upper[lock_from:n] = base[lock_from:]
    else:
        all_eq = storage_eq
        all_rhs = np.zeros(n)

    return {
        'p': p, 's': s, 'n': n, 'h0': h0, 'u0': u0, 'v0': v0, 'm': m,
        'obj': obj, 'lower': lower, 'upper': upper,
        'eq': all_eq, 'eq_rhs': all_rhs,
        'scenario_ub': scenario_ub, 'scenario_rhs': scenario_rhs,
        'mc': mc, 'md': md,
    }


def solve_lp_or_milp(parts):
    bounds = np.column_stack([parts['lower'], parts['upper']])
    lp = linprog(
        parts['obj'],
        A_ub=parts['scenario_ub'], b_ub=parts['scenario_rhs'],
        A_eq=parts['eq'], b_eq=parts['eq_rhs'],
        bounds=bounds, method='highs',
    )
    require(lp.success, 'LP 求解失败：' + lp.message)
    x = lp.x
    solver = 'LP'
    n, m = parts['n'], parts['m']

    simultaneous = (x[n:2*n] > CFG['tolerance']) & (x[2*n:3*n] > CFG['tolerance'])
    if simultaneous.any():
        t = np.arange(n)
        mode = coo_matrix(
            (
                np.r_[np.ones(n), -parts['mc']*np.ones(n), np.ones(n), parts['md']*np.ones(n)],
                (
                    np.r_[t, t, n+t, n+t],
                    np.r_[n+t, m+t, 2*n+t, m+t],
                ),
            ),
            shape=(2*n, m+n),
        ).tocsr()
        res = milp(
            np.r_[parts['obj'], np.zeros(n)],
            integrality=np.r_[np.zeros(m), np.ones(n)],
            bounds=Bounds(np.r_[parts['lower'], np.zeros(n)], np.r_[parts['upper'], np.ones(n)]),
            constraints=[
                LinearConstraint(hstack([parts['eq'], csr_matrix((parts['eq'].shape[0], n))]), parts['eq_rhs'], parts['eq_rhs']),
                LinearConstraint(hstack([parts['scenario_ub'], csr_matrix((parts['scenario_ub'].shape[0], n))]), -np.inf, parts['scenario_rhs']),
                LinearConstraint(mode, -np.inf, np.r_[np.zeros(n), np.full(n, parts['md'])]),
            ],
            options={'mip_rel_gap': 1e-8},
        )
        require(res.success, 'MILP 求解失败：' + res.message)
        x = res.x[:m]
        solver = 'MILP'

    residual = max(
        float(np.max(np.abs(parts['eq'] @ x - parts['eq_rhs']))),
        float(np.max(parts['scenario_ub'] @ x - parts['scenario_rhs'])),
        0.0,
    )
    require(residual < 1e-5, f'规划约束残差过大：{residual}')
    require(not (((x[n:2*n] > 1e-6) & (x[2*n:3*n] > 1e-6)).any()), '出现同时充放电')
    return x, solver, residual


def plan_day(net_scenarios, p, initial_kwh):
    parts = _storage_problem_parts(net_scenarios, p, initial_kwh)
    x, solver, residual = solve_lp_or_milp(parts)
    n = parts['n']
    return {
        'grid': np.maximum(x[:n], 0),
        'charge': np.maximum(x[n:2*n], 0),
        'discharge': np.maximum(x[2*n:3*n], 0),
        'storage': x[3*n:4*n+1],
        'solver': solver,
        'residual': residual,
    }


def adjust_next_six_hours(net_scenarios, p, initial_kwh, base_grid):
    n = len(base_grid)
    parts = _storage_problem_parts(
        net_scenarios, p, initial_kwh, fixed_grid=base_grid, lock_from=min(36, n),
    )
    x, solver, residual = solve_lp_or_milp(parts)
    return np.maximum(x[:n], 0), solver, residual


def execute_actual(grid, load_kwh, pv_kwh, initial_kwh, battery=True):
    n = len(grid)
    eta_c, eta_d = CFG['charge_efficiency'], CFG['discharge_efficiency']
    mc = CFG['max_charge_power_kw'] * CFG['interval_hours'] if battery else 0.0
    md = CFG['max_discharge_power_kw'] * CFG['interval_hours'] if battery else 0.0
    e = np.empty(n + 1)
    e[0] = initial_kwh
    charge = np.zeros(n)
    discharge = np.zeros(n)
    emergency = np.zeros(n)
    unused = np.zeros(n)
    for t in range(n):
        residual = grid[t] + pv_kwh[t] - load_kwh[t]
        if residual >= 0:
            charge[t] = min(residual, mc, max(0.0, (CFG['storage_max_kwh'] - e[t]) / eta_c))
            unused[t] = residual - charge[t]
        else:
            discharge[t] = min(-residual, md, max(0.0, (e[t] - CFG['storage_min_kwh']) * eta_d))
            emergency[t] = -residual - discharge[t]
        e[t+1] = e[t] + eta_c * charge[t] - discharge[t] / eta_d
    return {'charge': charge, 'discharge': discharge, 'emergency': emergency, 'unused': unused, 'storage': e}


def adjustment_fee(updated_grid, base_grid, p):
    increase = np.maximum(updated_grid - base_grid, 0.0)
    decrease = np.maximum(base_grid - updated_grid, 0.0)
    return p * (CFG['adjustment_up_multiplier'] * increase + CFG['adjustment_down_multiplier'] * decrease)


In [4]:
# 经济性回归：低概率高需求不应诱发免费超购；日内不应重复计原计划费。
def verify_purchase_cost_objective():
    original_cfg = CFG.copy()
    try:
        CFG['max_charge_power_kw'] = CFG['max_discharge_power_kw'] = 0.0
        CFG['planned_terminal_kwh'] = 6000.0
        scenarios = np.array([[100.0]] * 10 + [[1000.0]])
        p = np.ones(1)
        grid = plan_day(scenarios, p, 6000.0)['grid']
        updated, *_ = adjust_next_six_hours(np.array([[80.0]]), p, 6000.0, np.array([100.0]))
        # 购100度：期望费用100+5*900/11；购1000度：费用1000。
        assert np.allclose(grid, [100.0]), '零点规划遗漏普通购电费'
        expected_bill = float(p @ grid + 5 * np.maximum(scenarios-grid, 0).mean())
        assert np.isclose(expected_bill, 100 + 5*900/11)
        # 原计划100度、需求80度时，减购还要付费，保留计划才是最优。
        assert np.allclose(updated, [100.0]), '日内规划重复计入原计划购电费'
    finally:
        CFG.update(original_cfg)
    print('普通购电成本与日内结算回归检查通过。')

verify_purchase_cost_objective()


普通购电成本与日内结算回归检查通过。


## 3. 因果回测两种策略

In [5]:
def run_strategy(strategy_name, update_hours, progress_every=60):
    state = CFG['initial_storage_kwh']
    daily_rows = []
    detail_frames = []
    decision_rows = []

    for k, day in enumerate(dates):
        day_start_state = state
        if k == 0:
            base_plan = {'grid': np.zeros(144)}
        else:
            scenarios00 = scenario_net_load(k, start_slot=0)
            base_plan = plan_day(scenarios00, prices, state)
        base_grid = np.asarray(base_plan['grid'], float)
        updated_grid = base_grid.copy()
        blocks = []

        for start in (0, 36, 72, 108):
            end = start + 36
            issue_hour = start // 6
            if issue_hour in update_hours and k > 0:
                scenarios = scenario_net_load(k, start_slot=start)
                candidate, solver, residual = adjust_next_six_hours(
                    scenarios, prices[start:], state, base_grid[start:],
                )
                updated_grid[start:end] = candidate[:36]
                decision_rows.append({
                    '策略': strategy_name,
                    'date': day.date(),
                    '更新时间': f'{issue_hour:02d}:00',
                    '修改区间': f'{issue_hour:02d}:00-{issue_hour + 6:02d}:00',
                    '调整电量_kWh': float(np.abs(candidate[:36] - base_grid[start:end]).sum()),
                    '求解器': solver,
                    '约束残差': residual,
                })

            load_block = load_kw[k, start:end] * CFG['interval_hours']
            pv_block = pv_kw[k, start:end] * CFG['interval_hours']
            execution = execute_actual(
                updated_grid[start:end], load_block, pv_block, state, battery=(k > 0),
            )
            state = float(execution['storage'][-1])
            blocks.append(execution)

        charge = np.concatenate([x['charge'] for x in blocks])
        discharge = np.concatenate([x['discharge'] for x in blocks])
        emergency = np.concatenate([x['emergency'] for x in blocks])
        unused = np.concatenate([x['unused'] for x in blocks])
        storage_start = np.concatenate([x['storage'][:-1] for x in blocks])
        storage_end = np.concatenate([x['storage'][1:] for x in blocks])

        plan_fee = float(prices @ base_grid)
        adjustment_components = adjustment_fee(updated_grid, base_grid, prices)
        adjustment_total = float(adjustment_components.sum())
        emergency_fee = float(CFG['emergency_multiplier'] * (prices @ emergency))
        total_fee = float(plan_fee + adjustment_total + emergency_fee)

        daily_rows.append({
            '策略': strategy_name,
            'date': day.date(),
            '计划购电量(kWh)': float(base_grid.sum()),
            '最终计划购电量(kWh)': float(updated_grid.sum()),
            '增购量(kWh)': float(np.maximum(updated_grid - base_grid, 0).sum()),
            '减购量(kWh)': float(np.maximum(base_grid - updated_grid, 0).sum()),
            '紧急购电量(kWh)': float(emergency.sum()),
            '未利用电量(kWh)': float(unused.sum()),
            '计划费用(元)': plan_fee,
            '调整费用(元)': adjustment_total,
            '紧急费用(元)': emergency_fee,
            '总费用(元)': total_fee,
            '期初储能(kWh)': float(day_start_state),
            '期末储能(kWh)': float(state),
            '发生紧急购电': bool(emergency.sum() > 1e-6),
        })

        times = pd.date_range(day, periods=144, freq='10min')
        detail_frames.append(pd.DataFrame({
            '策略': strategy_name,
            'date': day.date(),
            'slot': np.arange(1, 145),
            'interval_start': times,
            'interval_end': times + pd.Timedelta(minutes=10),
            'price_yuan_per_kwh': prices,
            'actual_load_kwh': load_kw[k] * CFG['interval_hours'],
            'actual_pv_kwh': pv_kw[k] * CFG['interval_hours'],
            'planned_grid_kwh': base_grid,
            'updated_grid_kwh': updated_grid,
            'increase_kwh': np.maximum(updated_grid - base_grid, 0),
            'decrease_kwh': np.maximum(base_grid - updated_grid, 0),
            'charge_kwh': charge,
            'discharge_kwh': discharge,
            'emergency_kwh': emergency,
            'unused_kwh': unused,
            'storage_start_kwh': storage_start,
            'storage_end_kwh': storage_end,
            'planned_cost_yuan': prices * base_grid,
            'adjustment_cost_yuan': adjustment_components,
            'emergency_cost_yuan': CFG['emergency_multiplier'] * prices * emergency,
            'total_cost_yuan': prices * base_grid + adjustment_components + CFG['emergency_multiplier'] * prices * emergency,
            'evaluation': day.date() >= pd.Timestamp('2025-02-01').date(),
        }))

        if (k + 1) % progress_every == 0 or k == len(dates) - 1:
            print(f'{strategy_name}: {k + 1}/{len(dates)} 天完成')

    return pd.DataFrame(daily_rows), pd.concat(detail_frames, ignore_index=True), pd.DataFrame(decision_rows)


started = time.perf_counter()
results = {}
for strategy_name, update_hours in UPDATE_STRATEGIES.items():
    results[strategy_name] = run_strategy(strategy_name, update_hours)
print(f'两种策略回测完成，用时 {time.perf_counter() - started:.1f} 秒。')


对照：00时Q2计划不更新: 60/365 天完成


对照：00时Q2计划不更新: 120/365 天完成


对照：00时Q2计划不更新: 180/365 天完成


对照：00时Q2计划不更新: 240/365 天完成


对照：00时Q2计划不更新: 300/365 天完成


...

方案：06时+12时历史误差场景更新未来6h: 180/365 天完成


方案：06时+12时历史误差场景更新未来6h: 240/365 天完成


方案：06时+12时历史误差场景更新未来6h: 300/365 天完成


方案：06时+12时历史误差场景更新未来6h: 360/365 天完成


方案：06时+12时历史误差场景更新未来6h: 365/365 天完成
两种策略回测完成，用时 30.2 秒。


## 4. 校验并保存与 baseline 相同口径的明细

In [6]:
def verify_detail(detail):
    x = detail[detail['evaluation']].copy()
    balance = (
        x['updated_grid_kwh'] + x['actual_pv_kwh'] + x['discharge_kwh'] + x['emergency_kwh']
        - x['actual_load_kwh'] - x['charge_kwh'] - x['unused_kwh']
    )
    storage = (
        x['storage_end_kwh'] - x['storage_start_kwh']
        - CFG['charge_efficiency'] * x['charge_kwh']
        + x['discharge_kwh'] / CFG['discharge_efficiency']
    )
    checks = {
        '供需平衡最大残差': float(np.abs(balance).max()),
        '储能更新最大残差': float(np.abs(storage).max()),
        '储能下界违约': float(max(0.0, CFG['storage_min_kwh'] - x['storage_end_kwh'].min())),
        '储能上界违约': float(max(0.0, x['storage_end_kwh'].max() - CFG['storage_max_kwh'])),
        '充放电功率约束违约': float(max(0.0, x[['charge_kwh', 'discharge_kwh']].to_numpy().max() - CFG['max_charge_power_kw'] * CFG['interval_hours'])),
        '同时充放电时段数': int(((x['charge_kwh'] > 1e-6) & (x['discharge_kwh'] > 1e-6)).sum()),
        '费用分解最大残差': float(np.abs(x['total_cost_yuan'] - x[['planned_cost_yuan', 'adjustment_cost_yuan', 'emergency_cost_yuan']].sum(axis=1)).max()),
    }
    assert checks['供需平衡最大残差'] < 1e-5
    assert checks['储能更新最大残差'] < 1e-5
    assert checks['储能下界违约'] < 1e-5 and checks['储能上界违约'] < 1e-5
    assert checks['充放电功率约束违约'] < 1e-5
    assert checks['同时充放电时段数'] == 0
    assert checks['费用分解最大残差'] < 1e-5
    return checks


summaries = []
checks = {}
for name, (daily, detail, decisions) in results.items():
    daily_eval = daily[daily['date'] >= pd.Timestamp('2025-02-01').date()].copy()
    row = daily_eval[[
        '计划购电量(kWh)', '最终计划购电量(kWh)', '增购量(kWh)', '减购量(kWh)',
        '紧急购电量(kWh)', '未利用电量(kWh)', '计划费用(元)', '调整费用(元)',
        '紧急费用(元)', '总费用(元)'
    ]].sum().to_dict()
    row.update({
        '策略': name,
        '发生紧急购电天数': int(daily_eval['发生紧急购电'].sum()),
        '更新次数': 0 if decisions.empty else len(decisions[decisions['date'] >= pd.Timestamp('2025-02-01').date()]),
        '期初储能(kWh)': float(daily_eval['期初储能(kWh)'].iloc[0]),
        '期末储能(kWh)': float(daily_eval['期末储能(kWh)'].iloc[-1]),
    })
    summaries.append(row)
    checks[name] = verify_detail(detail)

summary = pd.DataFrame(summaries)
summary = summary[[
    '策略', '计划购电量(kWh)', '最终计划购电量(kWh)', '增购量(kWh)', '减购量(kWh)',
    '紧急购电量(kWh)', '未利用电量(kWh)', '计划费用(元)', '调整费用(元)', '紧急费用(元)',
    '总费用(元)', '发生紧急购电天数', '更新次数', '期初储能(kWh)', '期末储能(kWh)'
]]
base_cost = float(summary.loc[summary['策略'].eq('对照：00时Q2计划不更新'), '总费用(元)'].iloc[0])
summary['相对不更新节省(元)'] = base_cost - summary['总费用(元)']
summary['相对不更新节省比例'] = summary['相对不更新节省(元)'] / base_cost

for name, (daily, detail, decisions) in results.items():
    daily.to_csv(OUTPUT_DIR / f'{name}_daily_results.csv', index=False, encoding='utf-8-sig')
    detail.to_csv(OUTPUT_DIR / f'{name}_detail_10min.csv', index=False, encoding='utf-8-sig')
    decisions.to_csv(OUTPUT_DIR / f'{name}_decisions.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(checks).T.to_csv(OUTPUT_DIR / 'checks.csv', encoding='utf-8-sig')

print('物理约束与结算校验：')
display(pd.DataFrame(checks).T)
print('已保存到：', OUTPUT_DIR)


物理约束与结算校验：


,供需平衡最大残差,储能更新最大残差,储能下界违约,储能上界违约,充放电功率约束违约,同时充放电时段数,费用分解最大残差
对照：00时Q2计划不更新,2.273737e-13,9.094947e-13,0.0,0.0,0.0,0.0,0.0
方案：06时+12时历史误差场景更新未来6h,2.273737e-13,9.094947e-13,0.0,0.0,0.0,0.0,0.0


已保存到： /Users/jinyu/workspace/MCM/CUMCM_2026/.tmp/q3_01_outputs


## 5. 实际操作后的总费用

看 `总费用(元)` 与 `相对不更新节省(元)` 两列。该表也已保存为 `.tmp/q3_01_outputs/summary.csv`。

In [7]:
try:
    _summary = summary.copy()
except NameError:
    _summary = pd.read_csv(OUTPUT_DIR / 'summary.csv')

total_fee_result = _summary[[
    '策略', '总费用(元)', '计划费用(元)', '调整费用(元)', '紧急费用(元)',
    '紧急购电量(kWh)', '更新次数', '相对不更新节省(元)', '相对不更新节省比例'
]].copy()

display(total_fee_result.style.format({
    '总费用(元)': '{:,.2f}',
    '计划费用(元)': '{:,.2f}',
    '调整费用(元)': '{:,.2f}',
    '紧急费用(元)': '{:,.2f}',
    '紧急购电量(kWh)': '{:,.2f}',
    '相对不更新节省(元)': '{:,.2f}',
    '相对不更新节省比例': '{:.4%}',
}))
total_fee_result


,策略,总费用(元),计划费用(元),调整费用(元),紧急费用(元),紧急购电量(kWh),更新次数,相对不更新节省(元),相对不更新节省比例
0,对照：00时Q2计划不更新,"13,631,409.27","13,285,607.79",0.00,"345,801.47","55,283.75",0,0.00,0.0000%
1,方案：06时+12时历史误差场景更新未来6h,"14,618,373.22","13,281,388.02","1,197,102.55","139,882.64","21,778.47",668,"-986,963.95",-7.2404%


,策略,总费用(元),计划费用(元),调整费用(元),紧急费用(元),紧急购电量(kWh),更新次数,相对不更新节省(元),相对不更新节省比例
0,对照：00时Q2计划不更新,1.363141e+07,1.328561e+07,0.000000e+00,345801.473580,55283.753848,0,0.0000,0.000000
1,方案：06时+12时历史误差场景更新未来6h,1.461837e+07,1.328139e+07,1.197103e+06,139882.644489,21778.471936,668,-986963.9501,-0.072404
